# CareerAI: Integrated Recommender Systems (RS) + Reinforcement Learning (RL)

**Google Colab edition — run from top to bottom.**

## What this project demonstrates
- **RS:** resume text/skills → TF-IDF + skill-coverage hybrid ranking, matched/missing skills, ranking metrics.
- **RL:** target-conditioned career planning using tabular Q-learning, explicit states/actions/rewards/terminal conditions, learning curves.
- **Integration:** the recommender suggests suitable current roles; the user chooses a longer-term target; the RL planner finds a transition path.
- **Baselines:** compare hybrid ranking to TF-IDF-only and compare learned RL path with graph-search shortest path.

**Research honesty:** occupation data and benchmark relevance labels are curated/illustrative, not live vacancies or independently validated ground truth. Scores are not probabilities of hiring. RL rewards encode design assumptions, not proven real-world career outcomes.

## 1. Install dependencies
Run this cell first. Google Colab already includes most packages; this installs the PDF reader too.

In [ ]:
!pip -q install pypdf scikit-learn pandas numpy matplotlib networkx

In [ ]:
import io, re, random, textwrap, warnings
from collections import deque
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display, Markdown
warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
print('Environment ready')

## 2. Load the occupation dataset
The 20 occupation profiles below are a **curated prototype dataset**. You can replace them later with a properly sourced dataset such as O*NET/ESCO, subject to their terms.

In [ ]:
from pathlib import Path
from google.colab import files

# Try to load jobs.csv if you upload it. Otherwise, use the embedded dataset below.
CSV_DATA = r'''job_title,description,skills,level,next_roles
Data Analyst,"Analyze business data, create reports and dashboards, query databases, perform statistical analysis and communicate insights.","python;sql;excel;statistics;pandas;power bi;tableau;data analysis;data visualization",Entry,"Data Scientist;Business Analyst;Financial Analyst"
Business Analyst,"Translate business needs into requirements, analyze processes, prepare reports and work with stakeholders.","excel;sql;statistics;power bi;tableau;communication;requirements;business analysis;data visualization",Entry,"Product Manager;Data Analyst"
Financial Analyst,"Analyze financial statements, build forecasts, use spreadsheets and evaluate business and investment performance.","excel;statistics;financial analysis;accounting;forecasting;valuation;python;data analysis;communication",Entry,"Risk Analyst;Investment Analyst;Business Analyst"
Risk Analyst,"Measure financial and operational risks, analyze exposure, monitor risk indicators and prepare reports.","excel;sql;statistics;financial analysis;python;forecasting;communication;data analysis;mathematics",Entry,"Credit Risk Analyst;Quantitative Analyst;Financial Analyst"
Data Scientist,"Build predictive models, perform statistical analysis, engineer features and communicate data-driven findings.","python;sql;statistics;machine learning;pandas;numpy;scikit-learn;data visualization;experimentation",Mid,"Machine Learning Engineer;Research Scientist;Quantitative Analyst"
Machine Learning Engineer,"Deploy machine learning models, build data pipelines, test models and monitor production performance.","python;machine learning;deep learning;docker;api;git;cloud;testing;scikit-learn",Mid,"MLOps Engineer;AI Engineer;Data Scientist"
AI Engineer,"Build AI-powered applications, integrate models, evaluate outputs and deploy reliable AI systems.","python;machine learning;deep learning;api;cloud;git;testing;transformers;docker",Mid,"Machine Learning Engineer;MLOps Engineer;Research Scientist"
Research Scientist,"Conduct experiments, develop novel algorithms and publish research in machine learning and AI.","python;mathematics;statistics;machine learning;deep learning;pytorch;research;experimentation;algorithms",Advanced,"AI Engineer;Data Scientist"
Software Engineer,"Design, build, test and maintain software applications and APIs.","python;java;algorithms;data structures;api;git;testing;software engineering;sql",Entry,"Machine Learning Engineer;Cloud Engineer;Product Manager"
Cloud Engineer,"Design cloud infrastructure, manage deployments, automate operations and monitor reliability.","aws;azure;cloud;linux;networking;docker;kubernetes;automation;monitoring",Mid,"DevOps Engineer;MLOps Engineer;Cybersecurity Analyst"
DevOps Engineer,"Automate software delivery, manage infrastructure as code and improve deployment reliability.","linux;docker;kubernetes;ci/cd;automation;cloud;git;monitoring;networking",Mid,"Cloud Engineer;MLOps Engineer;Software Engineer"
MLOps Engineer,"Build and operate machine learning pipelines, model deployment, monitoring and reproducibility systems.","python;machine learning;docker;kubernetes;cloud;ci/cd;monitoring;automation;git",Mid,"Machine Learning Engineer;Cloud Engineer;AI Engineer"
Cybersecurity Analyst,"Monitor security events, investigate incidents and protect systems from threats.","linux;networking;cybersecurity;cryptography;incident response;security;monitoring;python",Entry,"Cloud Engineer;Security Engineer"
Security Engineer,"Design security controls, assess vulnerabilities and implement secure infrastructure and applications.","cybersecurity;linux;networking;cryptography;cloud;automation;security;python;incident response",Mid,"Cybersecurity Analyst;Cloud Engineer"
Product Manager,"Define product strategy, prioritize roadmaps, understand users and coordinate cross-functional delivery.","product management;communication;strategy;experimentation;analytics;business analysis;leadership;requirements",Mid,"Business Analyst;Data Analyst"
Marketing Analyst,"Analyze campaign performance, segment audiences and communicate marketing insights using data.","excel;statistics;analytics;data analysis;data visualization;sql;communication;experimentation",Entry,"Data Analyst;Business Analyst"
Investment Analyst,"Research investments, analyze companies and markets, build valuations and support portfolio decisions.","excel;financial analysis;accounting;forecasting;valuation;research;statistics;python",Entry,"Portfolio Analyst;Financial Analyst;Quantitative Analyst"
Portfolio Analyst,"Analyze portfolio performance, monitor risk and support asset allocation decisions.","financial analysis;statistics;valuation;forecasting;excel;python;risk analysis;research",Mid,"Investment Analyst;Quantitative Analyst"
Quantitative Analyst,"Apply mathematics, statistics and programming to financial modelling, pricing and risk analysis.","python;mathematics;statistics;algorithms;financial analysis;machine learning;sql;research",Mid,"Risk Analyst;Data Scientist;Investment Analyst"
Credit Risk Analyst,"Assess borrower creditworthiness, model default risk and monitor credit portfolios.","statistics;python;sql;financial analysis;risk analysis;excel;forecasting;data analysis",Mid,"Risk Analyst;Quantitative Analyst;Financial Analyst"
Portfolio Analyst,"Analyze portfolio performance, monitor risk and support asset allocation decisions.","financial analysis;statistics;valuation;forecasting;excel;python;risk analysis;research",Mid,"Investment Analyst;Quantitative Analyst"
'''

uploaded_jobs = None
try:
    # Optional upload: if jobs.csv is present in the Colab session, use it.
    if Path('/content/jobs.csv').exists():
        uploaded_jobs = pd.read_csv('/content/jobs.csv')
except Exception:
    pass
jobs = uploaded_jobs if uploaded_jobs is not None else pd.read_csv(io.StringIO(CSV_DATA))
jobs = jobs.drop_duplicates(subset=['job_title']).fillna('').reset_index(drop=True)
jobs['skill_list'] = jobs['skills'].apply(lambda x: sorted(set(s.strip().lower() for s in str(x).split(';') if s.strip())))
jobs['next_role_list'] = jobs['next_roles'].apply(lambda x: [s.strip() for s in str(x).split(';') if s.strip()])
print(f'Loaded {len(jobs)} occupation profiles')
display(jobs[['job_title','level','skills']].head(10))

## 3. Resume processing
Upload a text-based PDF resume or paste resume text. Extraction is rule-based and may miss synonyms, so review the detected skills. Avoid uploading sensitive information you do not need for this demo.

In [ ]:
SKILL_VOCAB = sorted(set(s for arr in jobs['skill_list'] for s in arr) | {
    'accounting','agile','algorithms','api','automation','aws','azure','c++','ci/cd','cloud','communication','cryptography','cybersecurity','data structures','deep learning','devops','docker','excel','experimentation','financial analysis','forecasting','git','github','incident response','java','javascript','kubernetes','leadership','linux','machine learning','mathematics','monitoring','networking','nlp','numpy','pandas','power bi','product management','python','pytorch','requirements','research','risk analysis','scikit-learn','security','software engineering','sql','statistics','strategy','tableau','tensorflow','testing','transformers','valuation'
})
ALIASES = {'powerbi':'power bi','scikit learn':'scikit-learn','ml':'machine learning','machine-learning':'machine learning','deep-learning':'deep learning','amazon web services':'aws','ci cd':'ci/cd','ci-cd':'ci/cd'}
def normalize_text(text):
    text = str(text or '').lower()
    for a,b in sorted(ALIASES.items(), key=lambda x: len(x[0]), reverse=True):
        text = text.replace(a,b)
    return re.sub(r'\s+',' ',text).strip()
def extract_skills(text):
    clean = normalize_text(text)
    found=[]
    for skill in SKILL_VOCAB:
        pattern = r'(?<![\w+#])' + re.escape(skill.lower()) + r'(?![\w+#])'
        if re.search(pattern, clean): found.append(skill.lower())
    return sorted(set(found))
def extract_pdf_text(pdf_bytes):
    from pypdf import PdfReader
    reader = PdfReader(io.BytesIO(pdf_bytes))
    return '\n'.join(page.extract_text() or '' for page in reader.pages)

resume_text = ''
choice = input('Type P to paste resume text, or U to upload a PDF: ').strip().lower()
if choice == 'u':
    uploaded = files.upload()
    if uploaded:
        fname = next(iter(uploaded))
        resume_text = extract_pdf_text(uploaded[fname])
else:
    print('Paste resume text below. Finish by entering a line containing END')
    lines=[]
    while True:
        line=input()
        if line.strip().upper()=='END': break
        lines.append(line)
    resume_text='\n'.join(lines)
if not resume_text.strip():
    print('No resume text provided; using a synthetic example profile for demonstration.')
    resume_text='Python SQL statistics pandas Excel machine learning data analysis financial analysis forecasting communication'
candidate_skills = extract_skills(resume_text)
print('Detected skills:', candidate_skills)
print('Review this list before relying on recommendations.')

## 4. Recommender System (RS)
We compare TF-IDF text similarity, explicit skill coverage, and a hybrid ranking. The hybrid score is a heuristic ranking score, **not** a hiring probability.

In [ ]:
def recommend_jobs(candidate_text, jobs_df, top_n=5, method='hybrid', preferred_level='Any', keyword=''):
    skills = extract_skills(candidate_text)
    docs = (jobs_df['job_title']+' '+jobs_df['description']+' '+jobs_df['skills'].str.replace(';',' ',regex=False)).tolist()
    candidate_doc = normalize_text(candidate_text)+' '+' '.join(skills)
    vec = TfidfVectorizer(stop_words='english', ngram_range=(1,2), sublinear_tf=True)
    mat = vec.fit_transform(docs+[candidate_doc])
    sims = cosine_similarity(mat[-1], mat[:-1]).ravel()
    coverage = [len(set(skills)&set(req))/max(1,len(req)) for req in jobs_df['skill_list']]
    out = jobs_df.copy()
    out['tfidf_score'] = sims
    out['skill_coverage'] = coverage
    if method == 'tfidf': out['match_score'] = 100*out['tfidf_score']
    elif method == 'skills': out['match_score'] = 100*out['skill_coverage']
    else: out['match_score'] = 100*(0.55*out['tfidf_score']+0.45*out['skill_coverage'])
    if preferred_level != 'Any': out['match_score'] += np.where(out['level'].eq(preferred_level),3,0)
    if keyword.strip():
        out['match_score'] += out.apply(lambda r: 3 if keyword.lower() in (r['job_title']+' '+r['description']).lower() else 0, axis=1)
    ranked = out.sort_values(['match_score','skill_coverage'],ascending=False).head(top_n).reset_index(drop=True)
    ranked.insert(0, 'rank', range(1, len(ranked) + 1))
    return ranked, skills

def skill_gap(job_row, possessed):
    required=set(job_row['skill_list']); have=set(possessed)
    return sorted(required & have), sorted(required-have)

preferred_level = input('Preferred experience level (Entry/Mid/Advanced/Any; press Enter for Any): ').strip().title() or 'Any'
if preferred_level not in ['Entry','Mid','Advanced','Any']: preferred_level='Any'
keyword = input('Optional keyword preference (e.g., finance, AI, cloud; press Enter to skip): ').strip()
ranked, candidate_skills = recommend_jobs(resume_text, jobs, top_n=10, method='hybrid', preferred_level=preferred_level, keyword=keyword)
show_cols=['job_title','level','match_score','tfidf_score','skill_coverage']
display(ranked[show_cols].style.format({'match_score':'{:.2f}','tfidf_score':'{:.3f}','skill_coverage':'{:.1%}'}))

if len(ranked):
    top = ranked.iloc[0]
    matched, missing = skill_gap(top, candidate_skills)
    display(Markdown(f"### Skill gap for top recommendation: {top['job_title']}\n**Matched skills:** {', '.join(matched) if matched else 'None detected'}\n\n**Skills to develop:** {', '.join(missing) if missing else 'No listed skill gaps'}"))

## 5. RS evaluation and baseline comparison
These benchmark labels are **rule-derived demonstration labels**, not independent ground truth. For a report, create manually reviewed held-out candidate profiles and have a rubric or subject expert label relevant jobs.

In [ ]:
benchmark_profiles = [
 ('analytics','Python SQL Excel statistics pandas data analysis dashboards', {'Data Analyst','Business Analyst','Marketing Analyst'}),
 ('finance','Excel financial analysis accounting forecasting valuation portfolio research', {'Financial Analyst','Investment Analyst','Portfolio Analyst'}),
 ('ml','Python machine learning deep learning pytorch tensorflow statistics data science', {'Data Scientist','Machine Learning Engineer','AI Engineer','Research Scientist'}),
 ('cloud','AWS Azure Docker Kubernetes Linux networking devops automation monitoring', {'Cloud Engineer','DevOps Engineer','MLOps Engineer'}),
 ('security','Linux networking cybersecurity cryptography incident response security', {'Cybersecurity Analyst','Security Engineer'})
]
def ranking_metrics(ranked_titles, relevant, k=5):
    top=ranked_titles[:k]
    rel=[1 if x in relevant else 0 for x in top]
    precision=sum(rel)/k
    recall=sum(rel)/max(1,len(relevant))
    dcg=sum(v/np.log2(i+2) for i,v in enumerate(rel))
    ideal=sum(1/np.log2(i+2) for i in range(min(k,len(relevant))))
    return precision, recall, (dcg/ideal if ideal else 0)
rows=[]
for name, txt, relevant in benchmark_profiles:
    for method in ['tfidf','skills','hybrid']:
        r,_=recommend_jobs(txt,jobs,top_n=len(jobs),method=method)
        p,rec,ndcg=ranking_metrics(r['job_title'].tolist(),relevant,5)
        rows.append({'profile':name,'method':method,'Precision@5':p,'Recall@5':rec,'NDCG@5':ndcg})
rs_results=pd.DataFrame(rows)
display(rs_results.round(3))
print('Reminder: labels are illustrative; interpret these values only as a pipeline demonstration.')

In [ ]:
ax=rs_results.groupby('method')[['Precision@5','Recall@5','NDCG@5']].mean().plot(kind='bar',figsize=(9,4),ylim=(0,1),title='RS baseline comparison (synthetic labels)')
ax.set_ylabel('Mean metric'); ax.set_xlabel('Ranking method'); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

## 6. Career transition graph and RL formulation
- **State:** current occupation (the compact prototype uses role as the state; candidate skills are used in reward shaping).
- **Action:** transition to a connected occupation.
- **Reward:** step cost + target-skill overlap + newly relevant skills + terminal goal bonus.
- **Terminal:** target reached, no legal actions, or maximum steps.

This is a simplified educational MDP. A stronger future version would include skill-learning actions and update the candidate state after each completed learning activity.

In [ ]:
def build_graph(jobs_df):
    known=set(jobs_df['job_title'])
    graph={r['job_title']:[d for d in r['next_role_list'] if d in known and d != r['job_title']] for _,r in jobs_df.iterrows()}
    return graph

def shortest_path(graph, start, goal):
    if start not in graph or goal not in graph: return []
    queue=deque([(start,[start])]); visited={start}
    while queue:
        node,path=queue.popleft()
        if node==goal: return path
        for nxt in graph.get(node,[]):
            if nxt not in visited:
                visited.add(nxt); queue.append((nxt,path+[nxt]))
    return []

def q_learning_plan(jobs_df, start_role, target_role, episodes=1500, alpha=.12, gamma=.92, epsilon=.25, seed=42, max_steps=12):
    graph=build_graph(jobs_df); states=list(graph); idx={s:i for i,s in enumerate(states)}
    if start_role not in graph or target_role not in graph: raise ValueError('Choose start and target roles from the dataset.')
    Q=np.zeros((len(states),len(states))); rng=np.random.default_rng(seed)
    skills_by={r['job_title']:set(r['skill_list']) for _,r in jobs_df.iterrows()}
    target_skills=skills_by[target_role]; rewards=[]; success=[]
    for ep in range(episodes):
        current=start_role; total=0.; reached=False
        for step in range(max_steps):
            if current==target_role: reached=True; break
            actions=graph.get(current,[])
            if not actions: break
            if rng.random()<epsilon: nxt=actions[int(rng.integers(len(actions)))]
            else:
                vals=[Q[idx[current],idx[a]] for a in actions]
                best=max(vals); tied=[a for a,v in zip(actions,vals) if np.isclose(v,best)]
                nxt=tied[int(rng.integers(len(tied)))]
            new_relevant=len((skills_by[nxt]-skills_by[current]) & target_skills)
            overlap=len(skills_by[nxt]&target_skills)/max(1,len(target_skills))
            reward=-1.0+2.0*overlap+0.25*new_relevant
            done=(nxt==target_role)
            if done: reward+=12.0
            next_actions=graph.get(nxt,[])
            future=0 if done or not next_actions else max(Q[idx[nxt],idx[a]] for a in next_actions)
            Q[idx[current],idx[nxt]] += alpha*(reward+gamma*future-Q[idx[current],idx[nxt]])
            total+=reward; current=nxt
            if done: reached=True; break
        rewards.append(total); success.append(int(reached))
    path=[start_role]; current=start_role; visited={current}
    for _ in range(max_steps):
        if current==target_role: break
        actions=[a for a in graph.get(current,[]) if a not in visited]
        if not actions: break
        vals=[Q[idx[current],idx[a]] for a in actions]
        current=actions[int(np.argmax(vals))]; path.append(current); visited.add(current)
    return {'path':path,'Q':Q,'states':states,'rewards':rewards,'success':success,'graph':graph,'target_reached':path[-1]==target_role}

roles=jobs['job_title'].tolist()
start_role = input(f'Start role (exact name; default {ranked.iloc[0]["job_title"]}): ').strip() or ranked.iloc[0]['job_title']
target_role = input('Target role (exact name; e.g. Machine Learning Engineer): ').strip() or 'Machine Learning Engineer'
if start_role not in roles: start_role=ranked.iloc[0]['job_title']; print('Unknown start role; using top RS recommendation:',start_role)
if target_role not in roles: target_role='Machine Learning Engineer'; print('Unknown target; using Machine Learning Engineer')
rl = q_learning_plan(jobs,start_role,target_role)
print('RL path:', ' → '.join(rl['path']))
print('Target reached:',rl['target_reached'])
base_path=shortest_path(rl['graph'],start_role,target_role)
print('Shortest-path baseline:', ' → '.join(base_path) if base_path else 'No path exists in the current graph')

In [ ]:
window=50
smooth=pd.Series(rl['rewards']).rolling(window,min_periods=1).mean()
fig,ax=plt.subplots(figsize=(9,4)); ax.plot(smooth); ax.set_title(f'Q-learning training reward (rolling mean, window={window})'); ax.set_xlabel('Episode'); ax.set_ylabel('Reward'); plt.tight_layout(); plt.show()
succ=pd.Series(rl['success']).rolling(window,min_periods=1).mean()
fig,ax=plt.subplots(figsize=(9,4)); ax.plot(succ); ax.set_title(f'Goal-reaching rate (rolling mean, window={window})'); ax.set_xlabel('Episode'); ax.set_ylabel('Success rate'); ax.set_ylim(0,1); plt.tight_layout(); plt.show()

## 7. Integrated recommendation + roadmap output
This combines the top RS recommendations, their skill gaps, and the RL-selected transition path. Remember that the RL role graph does not yet model course completion as a real state transition.

In [ ]:
display(Markdown('## Recommended roles'))
display(ranked[[c for c in ['rank','job_title','match_score','skill_coverage'] if c in ranked.columns]].head(5).round(3))
if len(ranked):
    print('Top role:',ranked.iloc[0]['job_title'])
    for _, row in ranked.head(3).iterrows():
        matched,missing=skill_gap(row,candidate_skills)
        print(f"\n{row['job_title']} | score={row['match_score']:.2f}")
        print('  Matched:', ', '.join(matched) if matched else 'None detected')
        print('  Missing:', ', '.join(missing) if missing else 'None from listed skills')
print('\nCareer roadmap:', ' → '.join(rl['path']))
if not rl['target_reached']:
    print('The agent did not reach the selected target. Inspect graph connectivity or choose another target/start pair.')

## 8. RL baseline experiment across reachable role pairs
For a fairer comparison, test multiple start/target pairs with paths in the graph. Q-learning and BFS answer slightly different objectives: BFS minimizes number of edges; Q-learning optimizes the chosen reward. Do not treat one as universally better without aligning the objectives.

In [ ]:
graph=build_graph(jobs)
all_pairs=[]
for s in graph:
    for t in graph:
        if s!=t:
            bp=shortest_path(graph,s,t)
            if bp: all_pairs.append((s,t,bp))
# Keep a manageable reproducible sample
sample_pairs=all_pairs[:min(12,len(all_pairs))]
comparison=[]
for i,(s,t,bp) in enumerate(sample_pairs):
    result=q_learning_plan(jobs,s,t,episodes=800,seed=SEED+i)
    comparison.append({'start':s,'target':t,'bfs_edges':len(bp)-1,'q_path_edges':len(result['path'])-1 if result['target_reached'] else np.nan,'q_reached':result['target_reached']})
rl_comparison=pd.DataFrame(comparison)
display(rl_comparison)
print('Q-learning target-reaching fraction in this sample:', round(rl_comparison['q_reached'].mean(),3) if len(rl_comparison) else 'n/a')

## 9. Conclusions, limitations, and next research steps
### What this prototype demonstrates
- A hybrid recommender that combines text similarity and skill coverage.
- Multiword skill extraction and explicit skill-gap reporting.
- A target-conditioned Q-learning planner over a career-transition graph.
- Basic RS ranking metrics, RL learning curves, and a graph-search baseline.

### Limitations to disclose
1. Occupation profiles and transitions are curated prototype data; not a live vacancy feed.
2. Benchmark relevance labels are rule-derived and not independent ground truth.
3. Rule-based resume parsing may miss synonyms and cannot establish proficiency.
4. The RL reward is a modelling assumption; the environment is not validated against real career outcomes.
5. The compact RL state is role-only; candidate skill updates are not yet fully represented as actions and state transitions.

### Next iteration
- Replace curated data with appropriately licensed O*NET/ESCO or a sourced job-description dataset.
- Manually label a held-out candidate-job relevance set and compare TF-IDF, skill coverage, hybrid and sentence embeddings.
- Expand the RL state to include skill mastery, learning time/budget and career preferences; add explicit learning actions.
- Compare Q-learning against shortest-path and reward-weighted graph search over multiple random seeds; report mean ± standard deviation.
- Add a user interface only after the experiment results are reproducible.

### Suggested subject mapping for your report/viva
**Recommender Systems:** content-based filtering, TF-IDF, cosine similarity, hybrid recommendation, skill matching, ranking metrics, cold-start handling and explainability.

**Reinforcement Learning:** MDP, state/action/reward, Q-table, epsilon-greedy exploration, learning rate α, discount factor γ, terminal states, cumulative reward, convergence diagnostics and baseline comparison.

**Integration:** RS selects relevant candidate roles; RL plans a target-conditioned sequence over the career graph; skill-gap analysis explains the steps.